In [1]:
!pip install pingouin
!pip install xlsxwriter

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import scipy
from scipy import stats

import numpy as np

import pandas as pd

from scipy.stats import kruskal
from scipy.stats import levene
from scipy.stats import f_oneway
import pingouin as pg

from scipy.stats import chisquare
from scipy.stats import chi2_contingency

import seaborn as sns

import xlsxwriter

In [4]:
#Open csv file.

data = pd.read_csv("/content/drive/MyDrive/TQP-AutoScore/final_data.csv", index_col = 0)

In [5]:
#Define groups and the modifiers.

group1 = 'Training'
group2 = 'Validation'
group3 = 'Test'

modifier = 'Dataset'

In [6]:
data['Dataset'].value_counts(normalize=False, dropna=False)

Training      190024
Test           63342
Validation     63341
Name: Dataset, dtype: int64

In [7]:
group1_value = 'Training'
group2_value = 'Validation'
group3_value = 'Test'

In [8]:
#See all columns.

print(list(data.columns))

['Age', 'Sex', 'Ethnicity', 'Weight', 'Height', 'Systolic Blood Pressure', 'Pulse Rate', 'Supplemental Oxygen', 'Pulse Oximetry', 'Respiratory Assistance', 'Respiratory Rate', 'Temperature', 'Pre-Hospital Cardiac Arrest', 'GCS - Eye', 'GCS - Verbal', 'GCS - Motor', 'Pupillary Response', 'Midline Shift', 'Substance Abuse Disorder', 'Diabetes Mellitus', 'Hypertension', 'Congestive Heart Failure', 'History of Myocardial Infarction', 'Angina Pectoris', 'History of Cerebrovascular Accident', 'Peripheral Arterial Disease', 'Chronic Obstructive Pulmonary Disease', 'Chronic Renal Failure', 'Cirrhosis', 'Bleeding Disorder', 'Disseminated Cancer', 'Currently Receiving Chemotherapy for Cancer', 'Dementia', 'Attention Deficit Disorder or Attention Deficit Hyperactivity Disorder', 'Mental or Personality Disorder', 'Ability to Complete Age-Appropriate ADL', 'Pregnancy', 'Anticoagulant Therapy', 'Steroid Use', 'Advanced Directive Limiting Care', 'Days from Incident to ED or Hospital Arrival', 'Transp

In [9]:
#Drop unwanted columns.

drop = []
data.drop(drop, axis=1, inplace=True)

In [10]:
#Define continuous and categorical columns.

continuous_columns = list(data.select_dtypes('number').columns)
print('Continuous columns: {}'.format(continuous_columns), '\n')

categorical_columns = list(data.select_dtypes('object').columns)
print('Categorical columns: {}'.format(categorical_columns))

Continuous columns: ['Age', 'Weight', 'Height', 'Systolic Blood Pressure', 'Pulse Rate', 'Pulse Oximetry', 'Respiratory Rate', 'Temperature', 'Days from Incident to ED or Hospital Arrival', 'AIS Severity - Maximum Severity of Injury in the Head', 'AIS Severity - Maximum Severity of Injury in the Face', 'AIS Severity - Maximum Severity of Injury in the Neck', 'AIS Severity - Maximum Severity of Injury in the Thorax', 'AIS Severity - Maximum Severity of Injury in the Abdomen', 'AIS Severity - Maximum Severity of Injury in the Spine', 'AIS Severity - Maximum Severity of Injury in the Upper Extremity', 'AIS Severity - Maximum Severity of Injury in the Lower Extremity', 'AIS Severity - Maximum Severity of Injury in Unspecified Body Regions', 'AIS derived ISS', 'Blood Transfusion', 'Alcohol Screen Result', 'Drug Screen - Cocaine', 'IMPACT Score', 'CRASH Score', 'label_oM', 'label_threedM', 'label_sevendM', 'label_fourteendM', 'label_thirtydM'] 

Categorical columns: ['Sex', 'Ethnicity', 'Sup

In [11]:
continuous_columns = ['Age', 'Weight', 'Height', 'Systolic Blood Pressure', 'Pulse Rate', 'Pulse Oximetry', 'Respiratory Rate', 'Temperature', 'Days from Incident to ED or Hospital Arrival', 'AIS Severity - Maximum Severity of Injury in the Head', 'AIS Severity - Maximum Severity of Injury in the Face', 'AIS Severity - Maximum Severity of Injury in the Neck', 'AIS Severity - Maximum Severity of Injury in the Thorax', 'AIS Severity - Maximum Severity of Injury in the Abdomen', 'AIS Severity - Maximum Severity of Injury in the Spine', 'AIS Severity - Maximum Severity of Injury in the Upper Extremity', 'AIS Severity - Maximum Severity of Injury in the Lower Extremity', 'AIS Severity - Maximum Severity of Injury in Unspecified Body Regions', 'AIS derived ISS', 'Blood Transfusion', 'Alcohol Screen Result', 'Drug Screen - Cocaine', 'IMPACT Score', 'CRASH Score']
categorical_columns = ['Sex', 'Ethnicity', 'Supplemental Oxygen', 'Respiratory Assistance', 'Pre-Hospital Cardiac Arrest', 'GCS - Eye', 'GCS - Verbal', 'GCS - Motor', 'Pupillary Response', 'Midline Shift', 'Substance Abuse Disorder', 'Diabetes Mellitus', 'Hypertension', 'Congestive Heart Failure', 'History of Myocardial Infarction', 'Angina Pectoris', 'History of Cerebrovascular Accident', 'Peripheral Arterial Disease', 'Chronic Obstructive Pulmonary Disease', 'Chronic Renal Failure', 'Cirrhosis', 'Bleeding Disorder', 'Disseminated Cancer', 'Currently Receiving Chemotherapy for Cancer', 'Dementia', 'Attention Deficit Disorder or Attention Deficit Hyperactivity Disorder', 'Mental or Personality Disorder', 'Ability to Complete Age-Appropriate ADL', 'Pregnancy', 'Anticoagulant Therapy', 'Steroid Use', 'Advanced Directive Limiting Care', 'Transport Mode', 'Inter-Facility Transfer', 'Trauma Type', 'Injury Intent', 'Mechanism of Injury', 'Work-Related', 'Neurosurgical Intervention', 'Alcohol Screen', 'Drug Screen - Amphetamine', 'Drug Screen - Barbiturate', 'Drug Screen - Benzodiazepines', 'Drug Screen - Cannabinoid', 'Drug Screen - MDMA or Ecstasy', 'Drug Screen - Methadone', 'Drug Screen - Methamphetamine', 'Drug Screen - Opioid', 'Drug Screen - Oxycodone', 'Drug Screen - Phencyclidine', 'Drug Screen - Tricyclic Antidepressant', 'ACS Verification Level', 'Hospital Type', 'Facility Bed Size', 'Primary Method of Payment', 'Race', 'Protective Device', 'Cerebral Monitoring', 'label_oM', 'label_threedM', 'label_sevendM', 'label_fourteendM', 'label_thirtydM', 'Dataset']


In [12]:
#Assign data to the groups.

x1 = data[data[modifier] == group1_value]
x2 = data[data[modifier] == group2_value]
x3 = data[data[modifier] == group3_value]

keys=[group1, group2, group3, 'Total']

# Continuous Data Description

In [13]:
#Create data description tables for continuous variables.

continuous_data_description = data[continuous_columns].describe()
continuous_data_description_x1 = x1[continuous_columns].describe()
continuous_data_description_x2 = x2[continuous_columns].describe()
continuous_data_description_x3 = x3[continuous_columns].describe()

In [14]:
#Calculate IQRs for total patient cohort.

iqrs = {}

for column in continuous_columns:
    q3, q1 = np.percentile(data[column], [75 ,25])
    iqr = q3 - q1
    iqrs[column] = iqr
    
iqrs = pd.DataFrame([iqrs], index=['iqr'])

continuous_data_description = continuous_data_description.append(iqrs)

In [15]:
#Calculate IQRs for Group 1.

iqrs_x1 = {}

for column in continuous_columns:
    q3, q1 = np.percentile(x1[column], [75 ,25])
    iqr = q3 - q1
    iqrs_x1[column] = iqr
    
iqrs_x1 = pd.DataFrame([iqrs_x1], index=['iqr'])

continuous_data_description_x1 = continuous_data_description_x1.append(iqrs_x1)

In [16]:
#Calculate IQRs for Group 2.

iqrs_x2 = {}

for column in continuous_columns:
    q3, q1 = np.percentile(x2[column], [75 ,25])
    iqr = q3 - q1
    iqrs_x2[column] = iqr
    
iqrs_x2 = pd.DataFrame([iqrs_x2], index=['iqr'])

continuous_data_description_x2 = continuous_data_description_x2.append(iqrs_x2)

In [17]:
#Calculate IQRs for Group 3.

iqrs_x3 = {}

for column in continuous_columns:
    q3, q1 = np.percentile(x3[column], [75 ,25])
    iqr = q3 - q1
    iqrs_x3[column] = iqr
    
iqrs_x3 = pd.DataFrame([iqrs_x3], index=['iqr'])

continuous_data_description_x3 = continuous_data_description_x3.append(iqrs_x3)

In [18]:
#Apply Shapiro–Wilk test to the total patient cohort for testing normality of the distribution,

shapiro_values = {}

for column in continuous_columns:
    statistic, p_value = scipy.stats.shapiro(data[column])
    shapiro_values[column] = p_value
    
shapiro_values = pd.DataFrame([shapiro_values], index=['shapiro_values'])

continuous_data_description = continuous_data_description.append(shapiro_values)

shapiro_values = shapiro_values.T

shapiro_values.loc[shapiro_values['shapiro_values'] > 0.05, 'shapiro_values'] = True
shapiro_values.loc[shapiro_values['shapiro_values'] != True, 'shapiro_values'] = False

/usr/local/lib/python3.8/dist-packages/scipy/stats/morestats.py:1760: UserWarning: p-value may not be accurate for N > 5000.
  warnings.warn("p-value may not be accurate for N > 5000.")


In [19]:
#Apply Shapiro–Wilk test to the Group 1 for testing normality of the distribution.

shapiro_values_x1 = {}

for column in continuous_columns:
    statistic, p_value = scipy.stats.shapiro(x1[column])
    shapiro_values_x1[column] = p_value
    
shapiro_values_x1 = pd.DataFrame([shapiro_values_x1], index=['shapiro_values'])

continuous_data_description_x1 = continuous_data_description_x1.append(shapiro_values_x1)

In [20]:
#Apply Shapiro–Wilk test to the Group 2 for testing normality of the distribution.

shapiro_values_x2 = {}

for column in continuous_columns:
    statistic, p_value = scipy.stats.shapiro(x2[column])
    shapiro_values_x2[column] = p_value
    
shapiro_values_x2 = pd.DataFrame([shapiro_values_x2], index=['shapiro_values'])

continuous_data_description_x2 = continuous_data_description_x2.append(shapiro_values_x2)

In [21]:
#Apply Shapiro–Wilk test to the Group 3 for testing normality of the distribution.

shapiro_values_x3 = {}

for column in continuous_columns:
    statistic, p_value = scipy.stats.shapiro(x3[column])
    shapiro_values_x3[column] = p_value
    
shapiro_values_x3 = pd.DataFrame([shapiro_values_x3], index=['shapiro_values'])

continuous_data_description_x3 = continuous_data_description_x3.append(shapiro_values_x3)

In [22]:
#Define normally and non-normally distributed variables for the total patient cohort.

normal_columns = shapiro_values[(shapiro_values['shapiro_values'] == True)]
normal_columns = list(normal_columns.index)

non_normal_columns = shapiro_values[(shapiro_values['shapiro_values'] == False)]
non_normal_columns = list(non_normal_columns.index)

In [23]:
#Apply Kruskal-Wallis test for non-normally distributed variables.

kw_values = {}

for column in non_normal_columns:
    statistic, p_value = stats.kruskal(x1[column], x2[column], x3[column])
    kw_values[column] = p_value
    
kw_values = pd.DataFrame([kw_values], index=['kw_values'])

continuous_data_description = continuous_data_description.append(kw_values)

In [24]:
#Apply Levene's test to assess the equality of variances for a normally distributed variables.

levene_values = {}

for column in normal_columns:
    statistic, p_value = stats.levene(x1[column], x2[column], x3[column])
    levene_values[column] = p_value
    
levene_values = pd.DataFrame([levene_values], index=['levene_values'])

continuous_data_description = continuous_data_description.append(levene_values)

levene_values = levene_values.T

levene_values.loc[levene_values['levene_values'] > 0.05, 'levene_values'] = True
levene_values.loc[levene_values['levene_values'] != True, 'levene_values'] = False

In [25]:
#Variables with equal and unequal variances for the total patient cohort.

unequal_variances = levene_values[(levene_values["levene_values"] == True)]
unequal_variances = list(unequal_variances.index)

equal_variances = levene_values[(levene_values["levene_values"] == False)]
equal_variances = list(equal_variances.index)

In [26]:
#Apply ANOVA test for normally distributed variables with equal variances.

anova_values = {}

for column in equal_variances:
    statistic, p_value = stats.f_oneway(x1[column], x2[column], x3[column])
    anova_values[column] = p_value
    
anova_values = pd.DataFrame([anova_values], index=['anova_values'])

continuous_data_description = continuous_data_description.append(anova_values)

In [27]:
#Apply Welch's ANOVA test for normally distributed variables with unequal variances.

wanova_values = {}

for column in unequal_variances:
    wanova = pg.welch_anova(dv=column, between=modifier, data=data)
    p_value = wanova.iloc[0]['p-unc']
    wanova_values[column] = p_value
    
wanova_values = pd.DataFrame([wanova_values], index=['wanova_values'])

continuous_data_description = continuous_data_description.append(wanova_values)

In [28]:
#Transpose data descriptions for dataframe merging.

continuous_data_description = continuous_data_description.T
continuous_data_description_x1 = continuous_data_description_x1.T
continuous_data_description_x2 = continuous_data_description_x2.T
continuous_data_description_x3 = continuous_data_description_x3.T

In [29]:
#Create an empty 'p Values' column.

p_values = np.nan

continuous_data_description['p Values'] = p_values

In [30]:
#Assign relevant p values to the 'p Values' for each variable.

for index, row in continuous_data_description.iterrows():
    if row['anova_values'] >= 0:
        row['p Values'] = row['anova_values']
    elif row['kw_values'] >= 0:
        row['p Values'] = row['kw_values']
    elif row['wanova_values'] >= 0:
        row['p Values'] = row['wanova_values']

In [31]:
#Create empty 'Mean or Median', 'Standard Deviation or Interquartile Range', 'Non-Normal Distribution', and 'Mean (±SD) or Median (IQR)' columns  (total patient cohort).

mean_median = np.nan
continuous_data_description['Mean or Median'] = mean_median

std_iqr = np.nan
continuous_data_description['Standard Deviation or Interquartile Range'] = std_iqr

normality = np.nan
continuous_data_description['Non-Normal Distribution'] = normality

msd_miqr = np.nan
continuous_data_description['Mean (±SD) or Median (IQR)'] = msd_miqr

In [32]:
#Assign mean for normally distributed variables and median for non-normally distributed variables to the 'Mean or Median' column  (total patient cohort).

for index, row in continuous_data_description.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Mean or Median'] = row['mean']
    elif row['shapiro_values'] < 0.05:
        row['Mean or Median'] = row['50%']

In [33]:
#Assign standard deviation for normally distributed variables and IQR for non-normally distributed variables to the 'Standard Deviation or Interquartile Range' column (total patient cohort).

for index, row in continuous_data_description.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Standard Deviation or Interquartile Range'] = row['std']
    elif row['shapiro_values'] < 0.05:
        row['Standard Deviation or Interquartile Range'] = row['iqr']

In [34]:
#Assign non-normally distributed variables their respective p-values for normality (total patient cohort).

for index, row in continuous_data_description.iterrows():
    if row['shapiro_values'] < 0.05:
        row['Non-Normal Distribution'] = row['shapiro_values']

In [35]:
#Format the data description table for continuous variables (total patient cohort).

continuous_data_description = continuous_data_description.drop(['count', 'mean', 'min', 'std', 'iqr', '25%', '50%', '75%', 'max', 'shapiro_values', 'kw_values', 'levene_values', 'anova_values', 'wanova_values'], axis=1)
continuous_data_description['p Values'] = continuous_data_description['p Values'].round(decimals = 3)
condition_1 = (continuous_data_description['p Values'] < 0.001)
continuous_data_description['p Values'] = np.where(condition_1 == True, ('<0.001'), (continuous_data_description['p Values'].astype(str)))
continuous_data_description['Mean or Median'] = continuous_data_description['Mean or Median'].round(decimals = 2)
continuous_data_description['Standard Deviation or Interquartile Range'] = continuous_data_description['Standard Deviation or Interquartile Range'].round(decimals = 2)
continuous_data_description['Non-Normal Distribution'] = continuous_data_description['Non-Normal Distribution'].replace(np.nan, 0)
condition_2 = (continuous_data_description['Non-Normal Distribution'] > 0)
continuous_data_description['Mean (±SD) or Median (IQR)'] = np.where(condition_2 == True,  (continuous_data_description['Mean or Median'].astype(str) + ' (' + continuous_data_description['Standard Deviation or Interquartile Range'].astype(str) + ')'), (continuous_data_description['Mean or Median'].astype(str) + ' (±' + continuous_data_description['Standard Deviation or Interquartile Range'].astype(str) + ')'))
continuous_data_description = continuous_data_description.drop(['Mean or Median', 'Non-Normal Distribution', 'Standard Deviation or Interquartile Range'], axis=1)
continuous_data_description = continuous_data_description[['Mean (±SD) or Median (IQR)', 'p Values']]

In [36]:
#Create empty 'Mean or Median', 'Standard Deviation or Interquartile Range', 'Non-Normal Distribution', and 'Mean (±SD) or Median (IQR)' columns  (Group 1).

mean_median = np.nan
continuous_data_description_x1['Mean or Median'] = mean_median

std_iqr = np.nan
continuous_data_description_x1['Standard Deviation or Interquartile Range'] = std_iqr

normality = np.nan
continuous_data_description_x1['Non-Normal Distribution'] = normality

msd_miqr = np.nan
continuous_data_description_x1['Mean (±SD) or Median (IQR)'] = msd_miqr

In [37]:
#Assign mean for normally distributed variables and median for non-normally distributed variables to the 'Mean or Median' column  (Group 1).

for index, row in continuous_data_description_x1.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Mean or Median'] = row['mean']
    elif row['shapiro_values'] < 0.05:
        row['Mean or Median'] = row['50%']

In [38]:
#Assign standard deviation for normally distributed variables and IQR for non-normally distributed variables to the 'Standard Deviation or Interquartile Range' column (Group 1).

for index, row in continuous_data_description_x1.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Standard Deviation or Interquartile Range'] = row['std']
    elif row['shapiro_values'] < 0.05:
        row['Standard Deviation or Interquartile Range'] = row['iqr']

In [39]:
#Assign non-normally distributed variables their respective p-values for normality (Group 1).

for index, row in continuous_data_description_x1.iterrows():
    if row['shapiro_values'] < 0.05:
        row['Non-Normal Distribution'] = row['shapiro_values']

In [40]:
#Format the data description table for continuous variables (Group 1).

continuous_data_description_x1 = continuous_data_description_x1.drop(['count', 'mean', 'min', 'std', 'iqr', '25%', '50%', '75%', 'max', 'shapiro_values'], axis=1)
continuous_data_description_x1['Mean or Median'] = continuous_data_description_x1['Mean or Median'].round(decimals = 2)
continuous_data_description_x1['Standard Deviation or Interquartile Range'] = continuous_data_description_x1['Standard Deviation or Interquartile Range'].round(decimals = 2)
continuous_data_description_x1['Non-Normal Distribution'] = continuous_data_description_x1['Non-Normal Distribution'].replace(np.nan, 0)
condition_1 =  (continuous_data_description_x1['Non-Normal Distribution'] > 0)
continuous_data_description_x1['Mean (±SD) or Median (IQR)'] = np.where(condition_1 == True,  (continuous_data_description_x1['Mean or Median'].astype(str) + ' (' + continuous_data_description_x1['Standard Deviation or Interquartile Range'].astype(str) + ')'), (continuous_data_description_x1['Mean or Median'].astype(str) + ' (±' + continuous_data_description_x1['Standard Deviation or Interquartile Range'].astype(str) + ')'))
continuous_data_description_x1 = continuous_data_description_x1.drop(['Mean or Median', 'Non-Normal Distribution', 'Standard Deviation or Interquartile Range'], axis=1)
continuous_data_description_x1 = continuous_data_description_x1[['Mean (±SD) or Median (IQR)']]

In [41]:
#Create empty 'Mean or Median', 'Standard Deviation or Interquartile Range', 'Non-Normal Distribution', and 'Mean (±SD) or Median (IQR)' columns  (Group 2).

mean_median = np.nan
continuous_data_description_x2['Mean or Median'] = mean_median

std_iqr = np.nan
continuous_data_description_x2['Standard Deviation or Interquartile Range'] = std_iqr

normality = np.nan
continuous_data_description_x2['Non-Normal Distribution'] = normality

msd_miqr = np.nan
continuous_data_description_x2['Mean (±SD) or Median (IQR)'] = msd_miqr

In [42]:
#Assign mean for normally distributed variables and median for non-normally distributed variables to the 'Mean or Median' column  (Group 2).

for index, row in continuous_data_description_x2.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Mean or Median'] = row['mean']
    elif row['shapiro_values'] < 0.05:
        row['Mean or Median'] = row['50%']

In [43]:
#Assign standard deviation for normally distributed variables and IQR for non-normally distributed variables to the 'Standard Deviation or Interquartile Range' column (Group 2).

for index, row in continuous_data_description_x2.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Standard Deviation or Interquartile Range'] = row['std']
    elif row['shapiro_values'] < 0.05:
        row['Standard Deviation or Interquartile Range'] = row['iqr']

In [44]:
#Assign non-normally distributed variables their respective p-values for normality (Group 2).

for index, row in continuous_data_description_x2.iterrows():
    if row['shapiro_values'] < 0.05:
        row['Non-Normal Distribution'] = row['shapiro_values']

In [45]:
#Format the data description table for continuous variables (Group 2).

continuous_data_description_x2 = continuous_data_description_x2.drop(['count', 'mean', 'min', 'std', 'iqr', '25%', '50%', '75%', 'max', 'shapiro_values'], axis=1)
continuous_data_description_x2['Mean or Median'] = continuous_data_description_x2['Mean or Median'].round(decimals = 2)
continuous_data_description_x2['Standard Deviation or Interquartile Range'] = continuous_data_description_x2['Standard Deviation or Interquartile Range'].round(decimals = 2)
continuous_data_description_x2['Non-Normal Distribution'] = continuous_data_description_x2['Non-Normal Distribution'].replace(np.nan, 0)
condition_1 =  (continuous_data_description_x2['Non-Normal Distribution'] > 0)
continuous_data_description_x2['Mean (±SD) or Median (IQR)'] = np.where(condition_1 == True,  (continuous_data_description_x2['Mean or Median'].astype(str) + ' (' + continuous_data_description_x2['Standard Deviation or Interquartile Range'].astype(str) + ')'), (continuous_data_description_x2['Mean or Median'].astype(str) + ' (±' + continuous_data_description_x2['Standard Deviation or Interquartile Range'].astype(str) + ')'))
continuous_data_description_x2 = continuous_data_description_x2.drop(['Mean or Median', 'Non-Normal Distribution', 'Standard Deviation or Interquartile Range'], axis=1)
continuous_data_description_x2 = continuous_data_description_x2[['Mean (±SD) or Median (IQR)']]

In [46]:
#Create empty 'Mean or Median', 'Standard Deviation or Interquartile Range', 'Non-Normal Distribution', and 'Mean (±SD) or Median (IQR)' columns  (Group 2).

mean_median = np.nan
continuous_data_description_x3['Mean or Median'] = mean_median

std_iqr = np.nan
continuous_data_description_x3['Standard Deviation or Interquartile Range'] = std_iqr

normality = np.nan
continuous_data_description_x3['Non-Normal Distribution'] = normality

msd_miqr = np.nan
continuous_data_description_x3['Mean (±SD) or Median (IQR)'] = msd_miqr

In [47]:
#Assign mean for normally distributed variables and median for non-normally distributed variables to the 'Mean or Median' column  (Group 3).

for index, row in continuous_data_description_x3.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Mean or Median'] = row['mean']
    elif row['shapiro_values'] < 0.05:
        row['Mean or Median'] = row['50%']

In [48]:
#Assign standard deviation for normally distributed variables and IQR for non-normally distributed variables to the 'Standard Deviation or Interquartile Range' column (Group 3).

for index, row in continuous_data_description_x3.iterrows():
    if row['shapiro_values'] > 0.05:
        row['Standard Deviation or Interquartile Range'] = row['std']
    elif row['shapiro_values'] < 0.05:
        row['Standard Deviation or Interquartile Range'] = row['iqr']

In [49]:
#Assign non-normally distributed variables their respective p-values for normality (Group 3).

for index, row in continuous_data_description_x3.iterrows():
    if row['shapiro_values'] < 0.05:
        row['Non-Normal Distribution'] = row['shapiro_values']

In [50]:
#Format the data description table for continuous variables (Group 3).

continuous_data_description_x3 = continuous_data_description_x3.drop(['count', 'mean', 'min', 'std', 'iqr', '25%', '50%', '75%', 'max', 'shapiro_values'], axis=1)
continuous_data_description_x3['Mean or Median'] = continuous_data_description_x3['Mean or Median'].round(decimals = 2)
continuous_data_description_x3['Standard Deviation or Interquartile Range'] = continuous_data_description_x3['Standard Deviation or Interquartile Range'].round(decimals = 2)
continuous_data_description_x3['Non-Normal Distribution'] = continuous_data_description_x3['Non-Normal Distribution'].replace(np.nan, 0)
condition_1 =  (continuous_data_description_x3['Non-Normal Distribution'] > 0)
continuous_data_description_x3['Mean (±SD) or Median (IQR)'] = np.where(condition_1 == True,  (continuous_data_description_x3['Mean or Median'].astype(str) + ' (' + continuous_data_description_x3['Standard Deviation or Interquartile Range'].astype(str) + ')'), (continuous_data_description_x3['Mean or Median'].astype(str) + ' (±' + continuous_data_description_x3['Standard Deviation or Interquartile Range'].astype(str) + ')'))
continuous_data_description_x3 = continuous_data_description_x3.drop(['Mean or Median', 'Non-Normal Distribution', 'Standard Deviation or Interquartile Range'], axis=1)
continuous_data_description_x3 = continuous_data_description_x3[['Mean (±SD) or Median (IQR)']]

In [51]:
#Merge data description tables for continuous variables.

continuous_results = pd.concat([continuous_data_description_x1, continuous_data_description_x2, continuous_data_description_x3, continuous_data_description], keys=keys, axis = 1)

In [52]:
#Save results for continuous variables.

continuous_results.to_csv('/content/drive/MyDrive/TQP-AutoScore/continuous_stats.csv')

# Categorical Data Description

In [53]:
#Remove the modifier for groups from categorical variables.

categorical_columns.remove(modifier)

In [54]:
#Apply chi-squared test for categorical variables.

chisq_test_values = {}

for column in categorical_columns:
    table = pd.crosstab(data[column], data[modifier], margins = False)
    stat, p, dof, expected = chi2_contingency(table)
    chisq_test_values[column] = p
    
chisq_test_values = pd.DataFrame([chisq_test_values], index=['p Values'])

categorical_columns_description = data[categorical_columns].describe()

In [55]:
#Transpose p values from the chi-squared test and format p values below 0.001 as '<0.001'.

chisq_test_values = chisq_test_values.T

chisq_test_values['p Values'] = chisq_test_values['p Values'].round(decimals = 3)
condition_1 = (chisq_test_values['p Values'] < 0.001)
chisq_test_values['p Values'] = np.where(condition_1 == True, ('<0.001'), (chisq_test_values['p Values'].astype(str)))

In [56]:
#Create data description tables for categorical variables with absolute values.

tables = {}

for column in categorical_columns:
    tables["{0}".format(column)] = pd.crosstab(index=data[column], columns=data[modifier])
    
categorical_data_description = pd.concat(tables, keys=categorical_columns)

categorical_data_description = categorical_data_description[[group1_value, group2_value, group3_value]]

In [57]:
#Create data description tables for categorical variables with frequencies.

tables = {}

for column in categorical_columns:
    tables["{0}".format(column)] = (pd.crosstab(index=data[column], columns=data[modifier], normalize = 'columns')*100).round(decimals = 2)
    
categorical_data_description_freq = pd.concat(tables, keys=categorical_columns)

categorical_data_description_freq = categorical_data_description_freq[[group1_value, group2_value, group3_value]]

In [58]:
#Merge data description tables for categorical variables.

categorical_data_description = pd.merge(categorical_data_description, categorical_data_description_freq, left_index=True, right_index=True, how='outer', sort = False)

In [59]:
#Create a 'total' column for the data description table for categorical variables.

x_column1 = '{}'.format(group1_value) + '_x'
x_column2 = '{}'.format(group2_value) + '_x'
x_column3 = '{}'.format(group3_value) + '_x'

y_column1 = '{}'.format(group1_value) + '_y'
y_column2 = '{}'.format(group2_value) + '_y'
y_column3 = '{}'.format(group3_value) + '_y'

patient_number = data.shape[0]

categorical_data_description['total'] = categorical_data_description[x_column1] + categorical_data_description[x_column2] + categorical_data_description[x_column3]

categorical_data_description['total frequencies'] = ((categorical_data_description['total']/patient_number)*100).round(decimals = 2)

categorical_data_description.reset_index(level = 1, inplace = True, col_level = 1)

In [60]:
#Append p values from chi squared test to the data description table for categorical variables. 

categorical_data_description = categorical_data_description.join(chisq_test_values)

categorical_data_description = categorical_data_description.rename_axis('Variables')

In [61]:
#Format the data description tables for categorical variables.

empty_column = np.nan
categorical_data_description['{}'.format(group1) +' n (%)'] = empty_column

empty_column = np.nan
categorical_data_description['{}'.format(group2) +' n (%)'] = empty_column

empty_column = np.nan
categorical_data_description['{}'.format(group3) +' n (%)'] = empty_column

empty_column = np.nan
categorical_data_description['Total, n (%)'] = empty_column

categorical_data_description['{}'.format(group1) +' n (%)'] = categorical_data_description[x_column1].astype(str) + ' (' + categorical_data_description[y_column1].round(decimals = 1).astype(str) + '%)'
categorical_data_description['{}'.format(group2) +' n (%)'] = categorical_data_description[x_column2].astype(str) + ' (' + categorical_data_description[y_column2].round(decimals = 1).astype(str) + '%)'
categorical_data_description['{}'.format(group3) +' n (%)'] = categorical_data_description[x_column3].astype(str) + ' (' + categorical_data_description[y_column3].round(decimals = 1).astype(str) + '%)'
categorical_data_description['Total, n (%)'] = categorical_data_description['total'].astype(str) + ' (' + categorical_data_description['total frequencies'].round(decimals = 1).astype(str) + '%)'
categorical_results = categorical_data_description[['level_1', '{}'.format(group1) +' n (%)', '{}'.format(group2) +' n (%)', '{}'.format(group3) +' n (%)', 'Total, n (%)', 'p Values']]

In [62]:
#Save results for categorical variables.

categorical_results.to_csv('/content/drive/MyDrive/TQP-AutoScore/categorical_stats.csv')

# Merge continuous and categorical results.

In [63]:
#Redifine the indexes of categorical results for merging purposes.

idx = 0
empty_col = np.nan
continuous_results.insert(loc=idx, column='level_1', value=empty_col)
continuous_results = continuous_results.rename_axis('Variables')
continuous_results.columns = [['Categorical Variable Values', group1, group2, group3, 'Total', ''], ['', 'Mean (±SD), Median (IQR), or n (%)', 'Mean (±SD), Median (IQR), or n (%)', 'Mean (±SD), Median (IQR), or n (%)', 'Mean (±SD), Median (IQR), or n (%)', 'p Values']]

categorical_results.columns = continuous_results.columns

In [64]:
#Merge categorical and continuous results.

results = pd.concat([categorical_results, continuous_results], axis=0)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
results

,Categorical Variable Values,Training,Validation,Test,Total,
,,"Mean (±SD), Median (IQR), or n (%)","Mean (±SD), Median (IQR), or n (%)","Mean (±SD), Median (IQR), or n (%)","Mean (±SD), Median (IQR), or n (%)",p Values
Variables,,,,,,
ACS Verification Level,Level I Trauma Center,93739 (49.3%),31240 (49.3%),31199 (49.2%),156178 (49.3%),0.907
ACS Verification Level,Level II Trauma Center,55172 (29.0%),18431 (29.1%),18424 (29.1%),92027 (29.1%),0.907
ACS Verification Level,Level III Trauma Center,1863 (1.0%),658 (1.0%),637 (1.0%),3158 (1.0%),0.907
ACS Verification Level,Unknown,39250 (20.7%),13012 (20.5%),13082 (20.6%),65344 (20.6%),0.907
Ability to Complete Age-Appropriate ADL,No,167834 (88.3%),55949 (88.3%),55969 (88.4%),279752 (88.3%),0.908
Ability to Complete Age-Appropriate ADL,Unknown,2574 (1.4%),869 (1.4%),886 (1.4%),4329 (1.4%),0.908
Ability to Complete Age-Appropriate ADL,Yes,19616 (10.3%),6523 (10.3%),6487 (10.2%),32626 (10.3%),0.908
Advanced Directive Limiting Care,No,187202 (98.5%),62391 (98.5%),62377 (98.5%),311970 (98.5%),0.785


In [65]:
#Reorder variables.

data_dictionary = pd.read_csv("/content/drive/MyDrive/TQP-AutoScore/Modified Data Dictionary.csv", encoding = 'latin1', index_col = None, low_memory = False)

field_names = list(data_dictionary['Field Name'])

variables = results.index

field_names = [x for x in field_names if x in variables]

#results = results.sort_values(by = ['Categorical Variable Values'])
results = results.iloc[pd.Categorical(results.index, field_names).argsort()]

In [66]:
#Save the final results table as a .csv file.

results.to_csv('/content/drive/MyDrive/TQP-AutoScore/stats.csv')

In [67]:
#Format and save the results as a .xlsx file.

df = pd.read_csv('/content/drive/MyDrive/TQP-AutoScore/stats.csv', header=None)
df.loc[0][0] = df.loc[2][0]
df.loc[0][6] = df.loc[1][6]
df.loc[1][6] = df.loc[1][1]
df = df.drop(2)
headers = df.iloc[0]
df  = pd.DataFrame(df.values[1:], columns=headers)

writer = pd.ExcelWriter('/content/drive/MyDrive/TQP-AutoScore/stats.xlsx', engine='xlsxwriter')
df.to_excel(writer, sheet_name='Sheet1', index=False)
workbook = writer.book
worksheet = writer.sheets['Sheet1']

for row in df['Variables'].unique():
    u=df.loc[df['Variables']==row].index.values + 1
    if len(u) <2: 
        pass
    else:
        worksheet.merge_range(u[0], 0, u[-1], 0, df.loc[u[0],'Variables'])
        
for row in df['Variables'].unique():
    u=df.loc[df['Variables']==row].index.values + 1
    if len(u) <2: 
        pass
    else:
        worksheet.merge_range(u[0], 6, u[-1], 6, df.loc[u[0],'p Values'])
        
cell_format = workbook.add_format()
cell_format.set_align('center')
cell_format.set_align('vcenter')
cell_format.set_text_wrap()

column_format = workbook.add_format()
column_format.set_bold(True)
column_format.set_align('center')
column_format.set_align('vcenter')
column_format.set_text_wrap()

worksheet.set_column('A:A', 20,column_format)
worksheet.set_column(1, 1, 17.86, cell_format)
worksheet.set_column(2, 5, 18.57, cell_format)
worksheet.set_column(6, 6, 7.86, cell_format)

worksheet.merge_range('A1:A2', 'Variables', column_format)
worksheet.merge_range('B1:B2', 'Categorical Variable Values', column_format)
worksheet.merge_range('G1:G2', 'p Values', column_format)

writer.save() 